# DL · Encuentro 4 · Detección, evaluación y revisión

Detección y decisiones con evidencia

**Tu copia personal:** en Colab elige **Archivo → Guardar una copia en Drive**. Conserva esa copia para los tres temas del encuentro.

Trabaja en CPU. Estas prácticas no requieren contratar GPU, claves de API ni servicios de pago. La primera preparación puede descargar bibliotecas ausentes; las prácticas de detección descargan los pesos del modelo si no están disponibles.

## Tu recorrido

1. [De una foto a un registro](#scrollTo=tema-dl-yolo)
2. [Una imagen marcada no prueba calidad](#scrollTo=tema-dl-evaluacion)
3. [Del modelo a una decisión responsable](#scrollTo=tema-dl-integracion)

Cada tema reúne **idea → caso explicado → preparación → ejecución → lectura y comprobación**. Puedes ejecutar todo el cuaderno o empezar por un tema: cada uno vuelve a preparar sus datos y herramientas. No mezcles las cifras de conjuntos diferentes.

Los datos inventados se identifican como simulaciones. Las fotos de detección son demostraciones y no incluyen una evaluación anotada de sus objetos. Las comprobaciones son voluntarias y sin nota.


## Tema 1 · De una foto a propuestas y registros

**Lo que vas a poder hacer:** Construir un registro de detecciones que pueda revisarse antes de contar objetos.

Ayuda a revisar un registro visual antes de tratarlo como inventario.

### Entiende la idea

Un detector analiza una foto y propone dónde puede haber objetos. Una propuesta incluye un nombre de categoría, un rectángulo que indica dónde mirar y una puntuación numérica. A ese rectángulo lo llamamos caja. YOLO (You Only Look Once; solo se mira una vez) es una familia de modelos que realiza detección. El umbral es el límite de puntuación que elegimos para mostrar o esconder propuestas. Superar ese límite no demuestra que el objeto esté bien identificado. Primero seguiremos una sola caja; después veremos cómo se obtienen varias propuestas de una foto en el cuaderno.

### Un caso pequeño antes del programa

**Una propuesta, distintos filtros**

En este ejercicio hay un objeto dibujado y una propuesta de caja con puntuación fija 0,80. Aplicamos la regla «conservar si puntuación ≥ umbral» sin mover la caja. La tabla muestra el filtro numérico; en el cuaderno YOLO ejecutarás detección sobre una foto.

| Umbral | ¿0,80 supera o iguala el límite? | Filas conservadas |
| --- | --- | --- |
| 0,50 | Sí | 1 |
| 0,80 | Sí | 1 |
| 0,90 | No | 0 |

1. Con 0,50 se conserva la propuesta: el registro contiene una fila con caja y puntuación.
2. Con 0,80 también se conserva, porque la regla incluye la igualdad.
3. Con 0,90 se descarta la fila, pero el objeto conocido sigue presente. Cero registros no demuestra ausencia.

**Qué demuestra:** El conteo cambia por una regla de selección, sin modificar la escena ni la puntuación.

### Ahora identifica los datos de esta práctica

Sigue una misma propuesta entre su caja, categoría, puntuación y fila del registro. No confundas filas conservadas con objetos confirmados.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** La biblioteca conserva puntuaciones estrictamente mayores al límite configurado. El esquema del aula incluye la igualdad: las dos reglas están identificadas.


### Nombres del modelo, los datos y el dispositivo

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **YOLO** — YOLO: You Only Look Once (Solo se mira una vez). Nombre de una familia de modelos de detección que propone categorías y cajas a partir de imágenes. El nombre nació con un enfoque que calcula esas propuestas en una evaluación de la red. Las variantes tienen arquitecturas y procesamiento propios; una caja conservada no demuestra que la propuesta sea correcta. [Fuente](https://arxiv.org/abs/1506.02640)

- **COCO** — COCO: Common Objects in Context (Objetos comunes en contexto). Conjunto de imágenes con anotaciones de objetos en escenas cotidianas. Sirve para desarrollar y comparar modelos de visión; no es un detector por sí mismo. El modelo usado en este cuaderno ofrece 80 categorías de ese conjunto, no cualquier objeto ni cualquier defecto agrícola. [Fuente](https://arxiv.org/abs/1405.0312)

- **CPU** — CPU: Central Processing Unit (Unidad central de procesamiento). Procesador que ejecuta las instrucciones generales de un programa. En los cuadernos, cpu identifica el dispositivo que realiza los cálculos. No es una categoría aprendida ni una medida de calidad. [Fuente](https://www.intel.com/content/www/us/en/products/docs/processors/cpu-vs-gpu.html)

- **RGB** — RGB: Red, Green, Blue (Rojo, verde y azul). Forma de representar un color mediante tres componentes o canales. Cada canal indica cuánto rojo, verde o azul participa en el color del píxel. La escala depende del formato: en imágenes de ocho bits suele ser 0–255 por canal; una preparación puede convertirla a 0–1. [Fuente](https://www.w3.org/TR/css-color-4/#rgb-functions)


In [ ]:
import sys, subprocess
from importlib import metadata

VERSION_ULTRALYTICS = '8.4.175'
try:
    version_instalada = metadata.version('ultralytics')
except metadata.PackageNotFoundError:
    version_instalada = None
if version_instalada != VERSION_ULTRALYTICS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           f'ultralytics=={VERSION_ULTRALYTICS}'])
import ultralytics
if ultralytics.__version__ != VERSION_ULTRALYTICS:
    raise RuntimeError('Ultralytics sigue cargado con otra versión. Reinicia el entorno y ejecuta de nuevo desde la preparación.')
print('Ultralytics:', ultralytics.__version__)


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, FileLink
import torch
from ultralytics import YOLO
from ultralytics.utils import ASSETS

SALIDA = Path('resultados_yolo')
SALIDA.mkdir(exist_ok=True)
torch.set_num_threads(2)
TRADUCCION = {'person':'persona','bus':'bus','car':'automóvil','truck':'camión',
              'bicycle':'bicicleta','motorcycle':'motocicleta','chair':'silla',
              'bottle':'botella','cup':'taza','laptop':'portátil','cell phone':'teléfono',
              'book':'libro','backpack':'mochila','tie':'corbata'}
def nombre_es(nombre):
    return TRADUCCION.get(nombre, nombre)
print('PyTorch:',torch.__version__)
print('Los nombres no traducidos conservan la categoría original en inglés.')


In [ ]:
modelo = YOLO('yolo26n.pt')
categorias = pd.DataFrame([{'id':i,'clase_original':nombre,'nombre':nombre_es(nombre)}
                           for i,nombre in modelo.names.items()])
print('Categorías conocidas:',len(categorias))
print('Ejemplos del catálogo:',', '.join(categorias.nombre.head(15)))
print('Son solo ejemplos; una categoría no mostrada aún puede existir. La búsqueda consulta el catálogo completo.')
BUSCAR_CATEGORIA='café' # después prueba 'persona' sin cambiar los pesos
consulta=BUSCAR_CATEGORIA.strip().casefold()
coincidencias=categorias[(categorias.nombre.str.casefold().str.contains(consulta,regex=False))|
                         (categorias.clase_original.str.casefold().str.contains(consulta,regex=False))] if consulta else categorias.iloc[:0]
if coincidencias.empty:
    print('No hay coincidencias para',repr(BUSCAR_CATEGORIA),'en este catálogo. No renombres otra clase para fingir que la reconoce.')
else:
    print('Coincidencias en todo el catálogo:',', '.join(coincidencias.nombre))


In [ ]:
foto = ASSETS / 'bus.jpg'
original = Image.open(foto).convert('RGB')
print('Ancho y alto:',original.size,'· canales: RGB')
fig,ax=plt.subplots(figsize=(5,7))
ax.imshow(original);ax.axis('off');ax.set_title('Entrada: la foto sin predicciones')
plt.tight_layout();plt.savefig(SALIDA/'01_original.png',dpi=130);plt.show()


In [ ]:
UMBRAL = 0.25
resultado = modelo.predict(source=str(foto), conf=UMBRAL, imgsz=640,
                           device='cpu', verbose=False)[0]
print('Detecciones conservadas:',len(resultado.boxes))
resultado.save(filename=str(SALIDA/'02_foto_detectada.jpg'))
fig,axs=plt.subplots(1,2,figsize=(10,7))
axs[0].imshow(original);axs[0].set_title('Antes: píxeles')
axs[1].imshow(resultado.plot()[:,:,::-1]);axs[1].set_title('Después: categorías, cajas y puntuaciones')
for ax in axs:ax.axis('off')
plt.tight_layout();plt.savefig(SALIDA/'03_antes_despues.png',dpi=130);plt.show()


In [ ]:
def tabla_detecciones(r):
    columnas=['deteccion','clase_id','clase_original','categoria','puntuacion',
              'x1','y1','x2','y2','ancho_px','alto_px']
    filas=[]
    for i,caja in enumerate(r.boxes, start=1):
        clase=int(caja.cls.item());nombre=r.names[clase]
        x1,y1,x2,y2=caja.xyxy[0].cpu().numpy().tolist()
        filas.append([i,clase,nombre,nombre_es(nombre),float(caja.conf.item()),
                      x1,y1,x2,y2,x2-x1,y2-y1])
    return pd.DataFrame(filas,columns=columnas)

tabla = tabla_detecciones(resultado)
display(tabla.round(3))
assert len(tabla)==len(resultado.boxes)
if not tabla.empty:
    primera=tabla.iloc[0]
    print('Primera detección:',primera.categoria,
          '· puntuación:',round(primera.puntuacion,3),
          '· tamaño:',round(primera.ancho_px),'×',round(primera.alto_px),'px')


In [ ]:
def resumir(t):
    return (t.groupby('categoria').size().rename('detecciones').reset_index()
             .sort_values('detecciones',ascending=False))
resumen=resumir(tabla)
display(resumen)
fig,ax=plt.subplots(figsize=(7,3))
if not resumen.empty:
    ax.bar(resumen.categoria,resumen.detecciones,color='#0075ad')
    ax.set_yticks(range(0,int(resumen.detecciones.max())+1))
else:ax.text(.5,.5,'Sin detecciones conservadas',ha='center',transform=ax.transAxes)
ax.set_ylabel('Detecciones en esta foto');ax.set_title('Un registro derivado de las cajas')
plt.tight_layout();plt.savefig(SALIDA/'04_conteo.png',dpi=130);plt.show()


In [ ]:
umbrales=[0.15,0.50,0.85]
comparacion=[]
fig,axs=plt.subplots(1,3,figsize=(12,6))
for ax,u in zip(axs,umbrales):
    r=modelo.predict(str(foto),conf=u,imgsz=640,device='cpu',verbose=False)[0]
    ax.imshow(r.plot()[:,:,::-1]);ax.axis('off')
    ax.set_title(f'Umbral {u:.2f}: {len(r.boxes)} detecciones')
    comparacion.append({'umbral':u,'detecciones':len(r.boxes)})
plt.tight_layout();plt.savefig(SALIDA/'05_umbrales.png',dpi=130);plt.show()
display(pd.DataFrame(comparacion))


### Los formatos del registro

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **CSV** — CSV: Comma-Separated Values (Valores separados por comas). Formato de texto que organiza registros con campos separados por comas. Los campos que contienen comas o saltos de línea pueden escribirse entre comillas; por eso un registro no siempre ocupa una sola línea visible. Puede incluir encabezados con nombres de columnas. Abrirlo como tabla no comprueba si sus datos o predicciones son correctos. [Fuente](https://www.rfc-editor.org/rfc/rfc4180)

- **JSON** — JSON: JavaScript Object Notation (Notación de objetos de JavaScript). Formato de texto que guarda datos estructurados con nombres, valores y listas. Aunque su nombre menciona JavaScript, otros lenguajes también pueden leerlo. Guardar la configuración ayuda a reconstruir una ejecución. [Fuente](https://www.rfc-editor.org/rfc/rfc8259)


In [ ]:
tabla.to_csv(SALIDA/'detecciones.csv',index=False,encoding='utf-8-sig')
resumen.to_csv(SALIDA/'conteo.csv',index=False,encoding='utf-8-sig')
config={'modelo':'yolo26n.pt','ultralytics':ultralytics.__version__,
        'torch':torch.__version__,'foto':foto.name,'umbral':UMBRAL,'imgsz':640,
        'dispositivo':'cpu','entrenamiento_en_este_cuaderno':False}
(SALIDA/'configuracion.json').write_text(json.dumps(config,indent=2,ensure_ascii=False))
for p in [SALIDA/'02_foto_detectada.jpg',SALIDA/'detecciones.csv',SALIDA/'conteo.csv',SALIDA/'configuracion.json']:
    print('Archivo creado:',p)
    display(FileLink(str(p)))


### Lee, comprueba y explica

Sigue una misma propuesta entre su caja, categoría, puntuación y fila del registro. No confundas filas conservadas con objetos confirmados.

**Prueba pequeña:** La biblioteca conserva puntuaciones estrictamente mayores al límite configurado. El esquema del aula incluye la igualdad: las dos reglas están identificadas.

**Vuelve al caso:** Relaciona cada fila con su imagen, categoría, puntuación y caja. Revisa las detecciones antes de convertir su conteo en una decisión operativa.

**Pregunta:** ¿Qué información necesita una fila de detección para poder interpretarla?

**Respuesta razonada:** La imagen de origen, categoría, puntuación, coordenadas y convención de la caja. Después hay que revisar omisiones y duplicados; la cantidad de filas no es automáticamente la cantidad real de objetos.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/dl-yolo)


## Tema 2 · Comprobar cajas con una referencia conocida

**Lo que vas a poder hacer:** Construir una comprobación de localización que separe filtro y coincidencia.

Una puntuación alta no garantiza una caja bien ubicada ni evita objetos omitidos.

### Entiende la idea

Evaluar un detector es comprobar sus propuestas contra respuestas de referencia. En una imagen de evaluación, una persona ha marcado qué objetos hay y dónde están; esas marcas son anotaciones. Para dar por correcta una propuesta comprobamos la categoría y cuánto coincide su caja con la anotada. IoU es una medida de esa coincidencia: compara la superficie compartida con toda la superficie cubierta por ambas cajas. Primero miraremos dos rectángulos. Después contaremos propuestas correctas, propuestas incorrectas y objetos que quedaron sin detectar. Una mala propuesta puede producir los dos últimos errores a la vez.

### Un caso pequeño antes del programa

**Conservar una caja no basta**

Un objeto de categoría Fruto tiene caja de referencia con esquina superior izquierda [20,20] e inferior derecha [60,60], en píxeles: ancho y alto de 40. La candidata tiene el mismo tamaño y categoría, y puntuación 0,80. Solo la desplazamos horizontalmente hacia la derecha. Aceptamos coincidencia si IoU ≥ 0,50; cada fila empieza un escenario independiente.

| Desplazamiento | Umbral de puntuación | IoU | TP / FP / FN |
| --- | --- | --- | --- |
| 10 píxeles | 0,50 | 0,60 | 1 / 0 / 0 |
| 30 píxeles | 0,50 | 1/7 ≈ 0,143 | 0 / 1 / 1 |
| 10 píxeles | 0,90 | 0,60 | 0 / 0 / 1 |

1. Con desplazamiento 10, comparten 30 × 40 = 1200 píxeles cuadrados; la unión mide 2000. IoU = 1200/2000 = 0,60.
2. Con desplazamiento 30, IoU = 400/2800 = 1/7: la propuesta cuenta como falsa y el objeto queda omitido.
3. Con umbral 0,90 se descarta la propuesta: no hay falso positivo, pero sigue habiendo un objeto omitido.

**Qué demuestra:** Filtrar y comprobar localización son decisiones distintas; ambas cambian los conteos.

### Ahora identifica los datos de esta práctica

La referencia sintética permite comprobar IoU y contar propuestas correctas, incorrectas y objetos omitidos. Separa el filtro de puntuación del criterio de ubicación.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Una caja mal ubicada puede producir una falsa propuesta y un objeto omitido al mismo tiempo. Este ejemplo no calcula mAP de bus.jpg.


### Nombres del modelo, los datos y el dispositivo

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **YOLO** — YOLO: You Only Look Once (Solo se mira una vez). Nombre de una familia de modelos de detección que propone categorías y cajas a partir de imágenes. El nombre nació con un enfoque que calcula esas propuestas en una evaluación de la red. Las variantes tienen arquitecturas y procesamiento propios; una caja conservada no demuestra que la propuesta sea correcta. [Fuente](https://arxiv.org/abs/1506.02640)

- **COCO** — COCO: Common Objects in Context (Objetos comunes en contexto). Conjunto de imágenes con anotaciones de objetos en escenas cotidianas. Sirve para desarrollar y comparar modelos de visión; no es un detector por sí mismo. El modelo usado en este cuaderno ofrece 80 categorías de ese conjunto, no cualquier objeto ni cualquier defecto agrícola. [Fuente](https://arxiv.org/abs/1405.0312)

- **CPU** — CPU: Central Processing Unit (Unidad central de procesamiento). Procesador que ejecuta las instrucciones generales de un programa. En los cuadernos, cpu identifica el dispositivo que realiza los cálculos. No es una categoría aprendida ni una medida de calidad. [Fuente](https://www.intel.com/content/www/us/en/products/docs/processors/cpu-vs-gpu.html)

- **RGB** — RGB: Red, Green, Blue (Rojo, verde y azul). Forma de representar un color mediante tres componentes o canales. Cada canal indica cuánto rojo, verde o azul participa en el color del píxel. La escala depende del formato: en imágenes de ocho bits suele ser 0–255 por canal; una preparación puede convertirla a 0–1. [Fuente](https://www.w3.org/TR/css-color-4/#rgb-functions)


In [ ]:
import sys, subprocess
from importlib import metadata

VERSION_ULTRALYTICS = '8.4.175'
try:
    version_instalada = metadata.version('ultralytics')
except metadata.PackageNotFoundError:
    version_instalada = None
if version_instalada != VERSION_ULTRALYTICS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           f'ultralytics=={VERSION_ULTRALYTICS}'])
import ultralytics
if ultralytics.__version__ != VERSION_ULTRALYTICS:
    raise RuntimeError('Ultralytics sigue cargado con otra versión. Reinicia el entorno y ejecuta de nuevo desde la preparación.')
print('Ultralytics:', ultralytics.__version__)


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, FileLink
import torch
from ultralytics import YOLO
from ultralytics.utils import ASSETS

SALIDA = Path('resultados_yolo')
SALIDA.mkdir(exist_ok=True)
torch.set_num_threads(2)
TRADUCCION = {'person':'persona','bus':'bus','car':'automóvil','truck':'camión',
              'bicycle':'bicicleta','motorcycle':'motocicleta','chair':'silla',
              'bottle':'botella','cup':'taza','laptop':'portátil','cell phone':'teléfono',
              'book':'libro','backpack':'mochila','tie':'corbata'}
def nombre_es(nombre):
    return TRADUCCION.get(nombre, nombre)
print('PyTorch:',torch.__version__)
print('Los nombres no traducidos conservan la categoría original en inglés.')


### DL4: siglas para comprobar las propuestas

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **IoU** — IoU: Intersection over Union (Intersección sobre unión). Medida de superposición entre dos regiones o cajas: área compartida dividida entre el área cubierta por ambas. La unión cuenta la zona compartida una sola vez. No tiene unidades: 0 significa sin área común y 1, coincidencia completa; no es la puntuación del detector. [Fuente](https://docs.ultralytics.com/guides/yolo-performance-metrics)

- **TP** — TP: True Positive (Verdadero positivo). Caso positivo correctamente identificado como positivo. Primero se declara qué significa positivo. En detección requiere emparejar una propuesta válida con un objeto de referencia. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

- **FP** — FP: False Positive (Falso positivo). Propuesta positiva que no corresponde a un positivo real. Es una falsa alerta. En detección cuenta una propuesta conservada sin coincidencia válida. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

- **FN** — FN: False Negative (Falso negativo). Caso realmente positivo que no se identifica correctamente. Es una omisión. En detección cuenta un objeto de referencia sin propuesta válida emparejada. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

- **AP** — AP: Average Precision (Precisión promedio). Resumen de la relación entre precisión y recall al recorrer los resultados ordenados por puntuación. Su cálculo exacto depende del protocolo de evaluación. En detección se fija el criterio de categoría, emparejamiento e IoU; AP no es la precisión en un único umbral. [Fuente](https://docs.ultralytics.com/guides/yolo-performance-metrics)

- **mAP** — mAP: mean Average Precision (Media de las precisiones promedio). Promedio de las AP de las categorías evaluadas según un protocolo. Algunas versiones también promedian resultados de varios criterios de IoU. Hay que indicar el protocolo: mAP50 usa IoU 0,50; mAP50–95 incluye IoU de 0,50 a 0,95 en pasos de 0,05. [Fuente](https://docs.ultralytics.com/modes/val)


In [ ]:
# Datos inventados para aprender evaluación, independientes de las fotos y de resultado.
from matplotlib.patches import Rectangle
def iou_cajas(a,b):
    izquierda=max(a[0],b[0]);arriba=max(a[1],b[1])
    derecha=min(a[2],b[2]);abajo=min(a[3],b[3])
    comun=max(0,derecha-izquierda)*max(0,abajo-arriba)
    area_a=max(0,a[2]-a[0])*max(0,a[3]-a[1])
    area_b=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    union=area_a+area_b-comun
    return comun/union if union else 0.

referencia=np.array([10.,20.,50.,60.]) # ancho 40 y alto 40 píxeles
PUNTUACION_EJEMPLO=.80;MINIMO_IOU=.50
escenarios=[('Desplazar 10 px',10,.50),('Desplazar 30 px',30,.50),('Filtrar con 0,90',10,.90)]
fig,axs=plt.subplots(1,3,figsize=(12,4));filas=[]
for ax,(nombre,desplazamiento,limite_puntuacion) in zip(axs,escenarios):
    propuesta=referencia+np.array([desplazamiento,0,desplazamiento,0])
    iou=iou_cajas(referencia,propuesta)
    conservada=PUNTUACION_EJEMPLO>limite_puntuacion
    tp=int(conservada and iou>=MINIMO_IOU);fp=int(conservada and not tp);fn=1-tp
    ax.add_patch(Rectangle(referencia[:2],40,40,color='#0075ad',alpha=.3,label='Referencia'))
    ax.add_patch(Rectangle(propuesta[:2],40,40,fill=False,edgecolor='#b85d00',lw=2,
                           linestyle='-' if conservada else '--',label='Propuesta' if conservada else 'Descartada'))
    comun_x=max(referencia[0],propuesta[0]);comun_ancho=max(0,min(referencia[2],propuesta[2])-comun_x)
    if comun_ancho:ax.add_patch(Rectangle((comun_x,20),comun_ancho,40,color='#198a87',alpha=.4,label='Área común'))
    ax.set(xlim=(0,95),ylim=(80,0),xlabel='x: píxeles desde la izquierda',ylabel='y: píxeles desde arriba')
    ax.set_aspect('equal');ax.set_title(f'{nombre}\nIoU={iou:.3f} · TP/FP/FN={tp}/{fp}/{fn}',fontsize=10);ax.legend(fontsize=7)
    filas.append([iou,tp,fp,fn])
    print(nombre,'· puntuación',PUNTUACION_EJEMPLO,'· límite',limite_puntuacion,
          '· conservada',conservada,'· aciertos/falsas propuestas/omisiones:',tp,fp,fn)
plt.tight_layout();plt.savefig(SALIDA/'06_evaluacion_cajas.png',dpi=130);plt.show()
assert np.isclose(filas[0][0],.6) and np.isclose(filas[1][0],1/7)
if MINIMO_IOU==.50 and PUNTUACION_EJEMPLO==.80:
    assert filas[0][1:]==[1,0,0] and filas[1][1:]==[0,1,1] and filas[2][1:]==[0,0,1]
else:
    print('Configuración cambiada: interpreta los nuevos conteos con el criterio que elegiste.')
print('IoU desplazamiento 10: área común 30×40=1200; unión 1600+1600−1200=2000; 1200/2000=0,60.')
print('IoU desplazamiento 30: área común 10×40=400; unión 1600+1600−400=2800; 400/2800=1/7.')


### Lee, comprueba y explica

La referencia sintética permite comprobar IoU y contar propuestas correctas, incorrectas y objetos omitidos. Separa el filtro de puntuación del criterio de ubicación.

**Prueba pequeña:** Una caja mal ubicada puede producir una falsa propuesta y un objeto omitido al mismo tiempo. Este ejemplo no calcula mAP de bus.jpg.

**Vuelve al caso:** Evalúa por separado la regla que conserva propuestas y el criterio que las compara con los objetos de referencia.

**Pregunta:** ¿Por qué dos cajas para un solo objeto no deben contarse como dos aciertos?

**Respuesta razonada:** El emparejamiento de evaluación asigna como máximo una detección al objeto de referencia. Una propuesta adicional sin otro objeto coincidente es un falso positivo; tampoco se borra una omisión por descartar una caja.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/dl-evaluacion)


## Tema 3 · Comparar costes de revisión

**Lo que vas a poder hacer:** Construir una política de revisión con costes y una salida verificable.

La decisión depende de errores y consecuencias, además de las puntuaciones.

### Entiende la idea

Una puntuación de la red no es una orden de trabajo. Hace falta una regla que convierta esa puntuación en una acción: revisar un grano o aceptarlo sin revisión. Esa regla es una política. Elegirla exige comparar consecuencias: cuánto cuesta revisar y cuánto cuesta dejar pasar un defecto. Coste significa aquí puntos supuestos para comparar, no precios reales. Usaremos los mismos diez granos y cambiaremos la regla; así veremos por qué la opción preferida depende de las consecuencias que declaramos y no solo de la red.

### Un caso pequeño antes del programa

**¿Qué granos conviene revisar?**

Diez granos y sus conteos ficticios del bloque 14. Enviamos a revisión los que alcanzan el umbral y aceptamos el resto. Revisar cuesta 1 unidad de coste; aceptar un defectuoso cuesta 8. Para esta cuenta suponemos que toda revisión detecta correctamente el defecto. Son unidades ficticias de coste, no precios.

| Política | Revisiones | Defectos aceptados | Coste total |
| --- | --- | --- | --- |
| Umbral 0,30 | 6 | 0 | 6 |
| Umbral 0,50 | 4 | 1 | 12 |
| Umbral 0,70 | 3 | 1 | 11 |
| Umbral 0,90 | 1 | 3 | 25 |
| Revisar todos | 10 | 0 | 10 |

1. Separa revisiones y defectos que pasan sin revisión.
2. Calcula coste = revisiones × 1 + defectos aceptados × 8. Por ejemplo, 4 × 1 + 1 × 8 = 12.
3. Con estos supuestos, 0,30 tiene el menor coste: 6 frente a 12, 11, 25 y 10.

**Qué demuestra:** Menos revisiones no implica menor coste cuando dejar pasar defectos resulta caro.

### Ahora identifica los datos de esta práctica

Compara revisar algunos granos con revisar todos. El coste combina cantidad de revisiones y defectos que se dejaron pasar.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Cambia solo el coste de un defecto aceptado de 8 a 1. La puntuación de la red no cambia; puede cambiar la decisión que prefieres.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### Referencias externas, fuera del recorrido de los encuentros

Los cuatro encuentros se resuelven con sus celdas principales y el cuaderno YOLO indicado. No requieren instalar ni ejecutar SAM, DINOv2, modelos de difusión de imágenes o modelos de lenguaje. Estas referencias son lectura opcional para otro momento; no sustituyen la práctica de las dos asignaturas.

Para ampliar por iniciativa propia, consulta las instrucciones y condiciones de los proyectos originales. No cambies a recursos de pago para completar este laboratorio. El pequeño Transformer del bloque 12 no es un modelo de conversación.


### Haz una cuenta antes de mirar la figura

Con límite 0,50 se revisan cuatro granos y se deja pasar un defecto. Revisar cuesta 1 por grano; omitir un defecto cuesta 8. Por tanto, 4×1 + 1×8 = **12 puntos**. Con límite 0,30 se revisan seis y no se omite ningún defecto: **6 puntos**. Revisar los diez costaría **10**.

Las puntuaciones y respuestas están dadas: **no vienen de la red entrenada**. Suponemos revisión perfecta para aislar la comparación. En una operación real habría que estudiar su capacidad y sus errores.

**Tip:** cambia solo COSTO_DEFECTO_ACEPTADO a 1 y ejecuta de nuevo. Comprueba que cambie la comparación aunque los diez granos y sus puntuaciones no cambien. La figura muestra puntos de coste; un punto bajo no demuestra ahorro comercial.


### 14A · Comparar reglas de revisión

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Positivo = defectuoso. Una puntuación alta dispara revisión manual.
real_defecto=np.array([0,1,0,1,1,0,0,1,0,0])
p_defecto=np.array([.1,.8,.2,.4,.9,.1,.6,.7,.2,.3])
COSTO_REVISION=1.;COSTO_DEFECTO_ACEPTADO=8.
filas=[]
for umbral in [.3,.5,.7,.9]:
    revisar=p_defecto>=umbral
    costo=revisar.sum()*COSTO_REVISION+np.sum((~revisar)&(real_defecto==1))*COSTO_DEFECTO_ACEPTADO
    filas.append([umbral,revisar.sum(),np.sum((~revisar)&(real_defecto==1)),costo])
resultado=pd.DataFrame(filas,columns=['Umbral','Granos revisados','Defectos omitidos','Costo ilustrativo']);display(resultado)
plt.plot(resultado.Umbral,resultado['Costo ilustrativo'],'o-',color=AZUL);plt.axhline(len(real_defecto)*COSTO_REVISION,color=VERDE,ls='--',label='Revisar todos');plt.xlabel('Umbral para revisar');plt.ylabel('Puntos de coste supuestos');plt.title('La decisión de negocio también tiene un parámetro');plt.legend();mostrar('14-impacto')
print('Supuesto del ejercicio: la revisión manual detecta todos los defectos. En un proyecto real, esa revisión también debe evaluarse.')


### Después de ejecutar: justifica la regla, no solo el mínimo

Con los costes iniciales, 0,30 obtiene 6 puntos y revisar todos 10. Explica qué revisión adicional evita qué omisión. Si omitir costara 1, los límites 0,70 y 0,90 empatan en 4 puntos: una decisión necesita también considerar capacidad y consecuencias, no solo una cifra. Cambiar un supuesto puede cambiar la opción preferida.


### Lee, comprueba y explica

Compara revisar algunos granos con revisar todos. El coste combina cantidad de revisiones y defectos que se dejaron pasar.

**Prueba pequeña:** Cambia solo el coste de un defecto aceptado de 8 a 1. La puntuación de la red no cambia; puede cambiar la decisión que prefieres.

**Vuelve al caso:** Compara políticas con los mismos casos, costes y supuestos. La opción elegida debe indicar qué se revisa, qué se acepta y cómo se seguirá el error.

**Pregunta:** ¿La arquitectura más compleja produce necesariamente la mejor decisión operativa?

**Respuesta razonada:** No. La elección depende de datos representativos, errores, coste, capacidad y tiempo de respuesta. La política elegida con validación debe comprobarse en una prueba separada y supervisarse cuando cambien las condiciones.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/dl-integracion)


## Cierra el encuentro con una explicación

1. **De una foto a un registro:** Relaciona cada fila con su imagen, categoría, puntuación y caja. Revisa las detecciones antes de convertir su conteo en una decisión operativa.
2. **Una imagen marcada no prueba calidad:** Evalúa por separado la regla que conserva propuestas y el criterio que las compara con los objetos de referencia.
3. **Del modelo a una decisión responsable:** Compara políticas con los mismos casos, costes y supuestos. La opción elegida debe indicar qué se revisa, qué se acepta y cómo se seguirá el error.

En cada resultado identifica entrada, salida, unidad o significado, forma de comprobarlo y límite. Guarda tu copia personal. Colab no registra automáticamente tu avance en el aula.
